# 05 - Create & publish the Fabric Data Agent

Run this **inside Fabric** (it uses the `fabric-data-agent-sdk`, which relies on the
Fabric runtime for auth and .NET). It creates/updates the Telco data agent, attaches
the Lakehouse, sets instructions + example queries, and publishes.

**Steps:** attach the Telco Lakehouse as the default Lakehouse, then **Run all**. Copy
the printed `DATA_AGENT_ARTIFACT_ID` and `DATA_AGENT_MCP_ENDPOINT` into your local `.env`.

> This is the supported way to create the data agent (it is not created from the local
> workstation). Config below mirrors `fabric/data-agent/config.yaml`, the source of truth.

In [ ]:
%pip install --quiet fabric-data-agent-sdk

## Configuration (mirrors fabric/data-agent/config.yaml)

In [ ]:
NAME = 'TelcoCustomerServiceAgent'
DESCRIPTION = 'Answers telecommunications customer-service questions over the Telco Lakehouse: billing and first-bill support, cross-sell/upsell, service health, outages, and retention/churn risk.\n'
LAKEHOUSE = 'TelcoLakehouse'
AI_INSTRUCTIONS = 'You are a data agent for a telecommunications customer-service team. You answer\nquestions about customers, accounts, billing, subscriptions, usage, service health,\noutages, work orders, offers, and churn/cross-sell scores.\n\nGuidance:\n- Prefer the curated gold tables (dim_*, fact_*, ml_*, customer_360).\n- customer_360 is a denormalized per-customer profile; use it for "give me a 360 view".\n- "First bill" means fact_invoice.is_first_bill = true. A confusing/high first bill\n  usually includes one-time activation and proration lines (fact_invoice_line).\n- "At-risk" / "churn risk" customers are in ml_churn_score (risk_band High/Medium/Low).\n- Cross-sell candidates and recommendations are in ml_crosssell_reco.\n- "Outage" or "service degradation" -> fact_outage (by geography) and fact_service_metric\n  (per-account latency/packet loss/uptime).\n- Always scope monetary answers to the correct account and time period.\n- Never invent values; only answer from the data.\n'
DS_INSTRUCTIONS = 'This Lakehouse holds synthetic telco data in a medallion layout. Query the\ncurated (gold) tables with clean names. Join keys: customer_id, account_id,\ngeo_id, product_id, plan_id, invoice_id, work_order_id.\n'
EXAMPLE_QUERIES = [('Show me the first bill details for account ACCT000123', "SELECT il.description, il.category, il.amount\nFROM fact_invoice i\nJOIN fact_invoice_line il ON i.invoice_id = il.invoice_id\nWHERE i.account_id = 'ACCT000123' AND i.is_first_bill = true\nORDER BY il.amount DESC\n"), ('Which new customers have an unpaid first bill?', 'SELECT customer_id, first_name, last_name, last_invoice_amount, open_balance\nFROM customer_360\nWHERE last_invoice_is_first_bill = true AND last_invoice_paid = false\n'), ('Which internet customers should be offered mobile?', "SELECT r.account_id, r.score, r.rationale\nFROM ml_crosssell_reco r\nWHERE r.recommended_product_id = 'PROD_MOB'\nORDER BY r.score DESC\n"), ('What products does customer CUST000045 already have?', "SELECT s.product_id, s.plan_id, s.mrc, s.status\nFROM fact_subscription s\nJOIN dim_account a ON s.account_id = a.account_id\nWHERE a.customer_id = 'CUST000045'\n"), ('Which active customers are high churn risk and had a recent outage?', "SELECT customer_id, first_name, last_name, churn_probability, churn_top_reason\nFROM customer_360\nWHERE risk_band = 'High' AND account_status = 'active'\n  AND recent_outage_exposure = true\nORDER BY churn_probability DESC\n"), ('What is the average uptime for account ACCT000123 over the last two weeks?', "SELECT ROUND(AVG(uptime_pct), 2) AS avg_uptime,\n       ROUND(AVG(latency_ms), 1) AS avg_latency_ms\nFROM fact_service_metric\nWHERE account_id = 'ACCT000123'\n")]

## Resolve the current workspace

In [ ]:
try:
    import notebookutils
    workspace_id = notebookutils.runtime.context.get('currentWorkspaceId')
except Exception:
    workspace_id = None
if not workspace_id:
    import sempy.fabric as fabric
    workspace_id = fabric.get_notebook_workspace_id()
print('Workspace:', workspace_id)

## Create, configure, and publish

In [ ]:
from fabric.dataagent.client import create_data_agent

agent = create_data_agent(data_agent_name=NAME, workspace_id=workspace_id)
agent.update_settings(ai_instructions=AI_INSTRUCTIONS)
print('Applied AI instructions.')

ds = agent.add_staging_datasource(
    artifact_name_or_id=LAKEHOUSE, workspace_id_or_name=workspace_id)
print('Added datasource:', LAKEHOUSE)

# datasource-level instructions (method name varies by SDK version)
for _m in ('update_configuration', 'update_settings', 'set_instructions'):
    _fn = getattr(ds, _m, None)
    if callable(_fn) and DS_INSTRUCTIONS:
        try:
            _fn(instructions=DS_INSTRUCTIONS)
            print('datasource instructions via', _m)
            break
        except Exception:
            pass

# example queries (best-effort across SDK versions)
_added = False
for _t in (ds, agent):
    for _m in ('add_example_queries', 'add_example_query'):
        _fn = getattr(_t, _m, None)
        if not callable(_fn):
            continue
        try:
            if _m.endswith('queries'):
                _fn(EXAMPLE_QUERIES)
            else:
                for _q, _s in EXAMPLE_QUERIES:
                    _fn(_q, _s)
            print('example queries via', _m)
            _added = True
            break
        except Exception:
            pass
    if _added:
        break

agent.publish_staging(description=DESCRIPTION)
print('Published data agent.')

## Data agent id + MCP endpoint

In [ ]:
aid = None
for _a in ('id', 'artifact_id', 'data_agent_id'):
    aid = getattr(agent, _a, None)
    if aid:
        break
if aid:
    print('DATA_AGENT_ARTIFACT_ID =', aid)
    print('DATA_AGENT_MCP_ENDPOINT =',
          f'https://api.fabric.microsoft.com/v1/mcp/workspaces/{workspace_id}/dataagents/{aid}/agent')
    print('\nAdd these to your local .env so the Foundry agents can bind to the data agent.')
else:
    print('Published. Copy the data agent id + MCP URL from the agent settings ->',
          'Model Context Protocol tab into .env.')